# CelebA-HQ DDPM training

Clones the repo from GitHub, downloads the Kaggle CelebA-HQ dataset to Google Drive, and trains with the same pipeline using the data loader loader.

## Mount Drive and clone repo

Update `REPO_URL` if needed.


In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/AAnanya19/Human-Faces-Generation-Diffusion-Models.git"
BRANCH = "main"
WORKDIR = Path("/content/Human-Faces-Generation-Diffusion-Models")

DRIVE_ROOT = Path("/content/drive/MyDrive/aml")
RUNS_ROOT = DRIVE_ROOT / "ddpm_runs"
RUN_NAME = "celebahq_run_001"
RUN_DIR = RUNS_ROOT / RUN_NAME

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"

assert WORKDIR.exists(), f"Repo folder not found: {WORKDIR}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", WORKDIR)
print("Run dir:", RUN_DIR)


## Install dependencies


In [ ]:
!pip install -q datasets huggingface_hub tqdm matplotlib torchvision kagglehub


## Kaggle dataset setup

This notebook uses `kagglehub.dataset_download(...)` and then copies the dataset into Drive the first time. Later runs reuse the Drive copy and skip the download.


In [ ]:
import kagglehub

print("kagglehub ready")


## Download CelebA-HQ to Drive

If the dataset has already been copied into Drive, this cell reuses it and skips the download.


In [ ]:
import shutil
from pathlib import Path

KAGGLE_DATASET = "badasstechie/celebahq-resized-256x256"
DATASET_ROOT = DRIVE_ROOT / "datasets"
DATASET_DIR = DATASET_ROOT / "celebahq_resized_256"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def count_images(root: Path) -> int:
    return sum(1 for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)

DATASET_ROOT.mkdir(parents=True, exist_ok=True)

if DATASET_DIR.exists() and count_images(DATASET_DIR) > 0:
    print("Using existing dataset from Drive:", DATASET_DIR)
else:
    cached_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print("Path to dataset files:", cached_path)
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    shutil.copytree(cached_path, DATASET_DIR)
    print("Copied dataset to Drive:", DATASET_DIR)

image_count = count_images(DATASET_DIR)
print("Dataset directory:", DATASET_DIR)
print("Image files found:", image_count)
if image_count == 0:
    raise RuntimeError("No images found after download. Inspect DATASET_DIR contents.")


## Training config


In [ ]:
EPOCHS = 50
BATCH_SIZE = 16
IMAGE_SIZE = 64
LR = 1e-4
TIMESTEPS = 1000
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
SAMPLE_EVERY = 50
NUM_SAMPLE_IMAGES = 8
CHECKPOINT_EVERY = 50
FOLDER_SUBSET_SIZE = 3000
FOLDER_TEST_SIZE = 300
BASE_CHANNELS = 64
TIME_DIM = 256
CHANNEL_MULTS = "1,2,4,8"
NUM_RES_BLOCKS = 2
DROPOUT = 0.1
ATTENTION_RESOLUTIONS = "16,8"
FIXED_SAMPLE_SEED = 123
FIXED_TRAJECTORY_SEED = 321
TRAJECTORY_SAVE_EVERY = 100


## Run training


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", device)
if device != "cuda":
    raise RuntimeError("GPU is not active. In Colab, switch Runtime -> Change runtime type -> GPU.")

!python3 /content/Human-Faces-Generation-Diffusion-Models/src/training/train.py \
    --epochs $EPOCHS \
    --batch_size $BATCH_SIZE \
    --image_size $IMAGE_SIZE \
    --lr $LR \
    --timesteps $TIMESTEPS \
    --weight_decay $WEIGHT_DECAY \
    --grad_clip $GRAD_CLIP \
    --sample_every $SAMPLE_EVERY \
    --num_sample_images $NUM_SAMPLE_IMAGES \
    --checkpoint_every $CHECKPOINT_EVERY \
    --dataset_source folder \
    --dataset_path "$DATASET_DIR" \
    --folder_subset_size $FOLDER_SUBSET_SIZE \
    --folder_test_size $FOLDER_TEST_SIZE \
    --base_channels $BASE_CHANNELS \
    --time_dim $TIME_DIM \
    --channel_mults $CHANNEL_MULTS \
    --num_res_blocks $NUM_RES_BLOCKS \
    --dropout $DROPOUT \
    --attention_resolutions $ATTENTION_RESOLUTIONS \
    --fixed_sample_seed $FIXED_SAMPLE_SEED \
    --fixed_trajectory_seed $FIXED_TRAJECTORY_SEED \
    --trajectory_save_every $TRAJECTORY_SAVE_EVERY \
    --device cuda \
    --save_dir "$RUN_DIR"


## Inspect saved outputs


In [ ]:
print("Checkpoint files:", sorted(str(p.name) for p in RUN_DIR.glob("*.pth")))
print("Sample grids:", sorted(str(p.name) for p in (RUN_DIR / "generated_samples").glob("*.png")))
print("Trajectory grids:", sorted(str(p.name) for p in (RUN_DIR / "trajectories").glob("*.png")))
print("Loss log exists:", (RUN_DIR / "loss_log.csv").is_file())
print("Metadata file exists:", (RUN_DIR / "run_config.json").is_file())
print((RUN_DIR / "run_config.json").read_text())
